# Day 16 Tutorial：用 NumPy 手写 MLP 前向传播

## Goal

实现“线性层→ReLU→线性输出”，保存每个中间量的 shape 与数值范围，并证明前向调用不会修改随机初始化参数。


## Setup

固定 seed 生成小权重。权重没有利用标签学习，所有输出只用于理解数据流，不能作为模型性能。


In [1]:
import platform
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print({"python": platform.python_version(), "numpy": np.__version__, "seed": RANDOM_SEED})


{'python': '3.10.20', 'numpy': '2.2.6', 'seed': 42}


## Steps

### 1. 声明输入与两层参数


In [2]:
X = np.array([
    [1.0, 0.0, 2.0],
    [0.5, 1.0, 1.5],
    [2.0, 1.0, 0.0],
    [-1.5, 0.5, 1.0],
    [0.0, -1.0, 2.0],
])
INPUT_FEATURES = X.shape[1]
HIDDEN_UNITS = 4
OUTPUTS = 1

W1 = rng.normal(scale=0.4, size=(INPUT_FEATURES, HIDDEN_UNITS))
b1 = np.zeros(HIDDEN_UNITS)
W2 = rng.normal(scale=0.4, size=(HIDDEN_UNITS, OUTPUTS))
b2 = np.zeros(OUTPUTS)

parameter_shapes = {"W1": W1.shape, "b1": b1.shape, "W2": W2.shape, "b2": b2.shape}
print(parameter_shapes)


{'W1': (3, 4), 'b1': (4,), 'W2': (4, 1), 'b2': (1,)}


### 2. 前向函数显式接收参数


In [3]:
def relu(values):
    return np.maximum(0.0, values)

def forward(inputs, W1, b1, W2, b2):
    z1 = inputs @ W1 + b1
    h1 = relu(z1)
    prediction = h1 @ W2 + b2
    return z1, h1, prediction

W1_before = W1.copy()
b1_before = b1.copy()
W2_before = W2.copy()
b2_before = b2.copy()

z1, h1, prediction = forward(X, W1, b1, W2, b2)
print("First three random predictions (not trained):")
display(pd.DataFrame(prediction, columns=["prediction"]).head(3).round(4))


First three random predictions (not trained):


,prediction
0,-0.1526
1,-0.0453
2,-0.0933


### 3. 建立前向 trace 表


In [4]:
def trace_row(name, values, meaning):
    return {
        "name": name,
        "shape": str(values.shape),
        "minimum": float(values.min()),
        "maximum": float(values.max()),
        "zero_fraction": float(np.mean(values == 0.0)),
        "meaning": meaning,
    }

trace = pd.DataFrame([
    trace_row("X", X, "input samples × features"),
    trace_row("z1", z1, "hidden pre-activation"),
    trace_row("h1", h1, "hidden output after ReLU"),
    trace_row("prediction", prediction, "random forward output"),
])
display(trace.round(4))


,name,shape,minimum,maximum,zero_fraction,meaning
0,X,"(5, 3)",-1.5000,2.0000,0.20,input samples × features
1,z1,"(5, 4)",-1.3529,1.0037,0.00,hidden pre-activation
2,h1,"(5, 4)",0.0000,1.0037,0.45,hidden output after ReLU
3,prediction,"(5, 1)",-0.1526,0.0101,0.00,random forward output


### 4. 对照没有非线性的输出


In [5]:
linear_only_prediction = z1 @ W2 + b2
comparison = pd.DataFrame({
    "with_relu": prediction.ravel(),
    "without_relu": linear_only_prediction.ravel(),
    "difference": prediction.ravel() - linear_only_prediction.ravel(),
})
display(comparison.round(4))


,with_relu,without_relu,difference
0,-0.1526,-0.6479,0.4953
1,-0.0453,-0.6240,0.5787
2,-0.0933,-0.7175,0.6242
3,0.0101,0.0899,-0.0798
4,-0.1151,-0.1879,0.0728


## Checks

检查所有 shape、ReLU 规则和参数只读性。


In [6]:
assert z1.shape == (len(X), HIDDEN_UNITS)
assert h1.shape == z1.shape
assert prediction.shape == (len(X), OUTPUTS)
assert np.all(h1 >= 0.0)
assert np.array_equal(h1, np.maximum(0.0, z1))
assert np.array_equal(W1, W1_before)
assert np.array_equal(b1, b1_before)
assert np.array_equal(W2, W2_before)
assert np.array_equal(b2, b2_before)
assert (z1 < 0).any(), "Tutorial expects at least one negative pre-activation"

print("Checks passed: forward shapes are correct and parameters did not change.")


Checks passed: forward shapes are correct and parameters did not change.


## Next Steps

完成手算后进入 Day 17：用标签计算损失，再根据梯度更新参数。当前随机预测即使偶然接近某个标签，也没有训练或泛化证据。
